In [1]:
pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00


In [2]:
import io
import json
import hashlib
from typing import Dict, List, Iterable, Optional

import boto3
import pandas as pd
from tqdm import tqdm
from botocore.config import Config
from botocore.exceptions import ClientError
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

try:
    from google.colab import userdata
except ImportError:
    userdata = None


# =========================
# CONFIG
# =========================
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-2"

SOURCE_BUCKET = "cleaned-legal-data-analysis-src-v2"

# MAUD source prefixes
# latest first, then whole cleaned_sections tree
SOURCE_PREFIXES = [
    "MAUD_Dataset/cleaned_sections/latest/",
    "MAUD_Dataset/cleaned_sections/",
]

TARGET_VECTOR_BUCKET = "combined.embeddings"
TARGET_INDEX_NAME = "legal-combined-index"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_DIMENSION = 384
DISTANCE_METRIC = "cosine"
DATA_TYPE = "float32"

MAX_TOKENS = 350
OVERLAP_TOKENS = 75

EMBED_BATCH_CHUNKS = 256
PUT_BATCH_SIZE = 25
GET_EXISTING_BATCH_SIZE = 100

ALLOWED_SUFFIXES = (".parquet", ".jsonl", ".json")


# =========================
# CLIENTS
# =========================
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

cfg = Config(retries={"max_attempts": 10, "mode": "standard"})
s3 = session.client("s3", config=cfg)
s3vectors = session.client("s3vectors", config=cfg)


# =========================
# HELPERS
# =========================
def list_s3_keys(bucket: str, prefix: str) -> Iterable[str]:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            lower = key.lower()
            if lower.endswith(ALLOWED_SUFFIXES):
                yield key


def read_s3_object(bucket: str, key: str) -> bytes:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read()


def read_dataframe_from_s3(bucket: str, key: str) -> Optional[pd.DataFrame]:
    raw = read_s3_object(bucket, key)
    lower = key.lower()

    if lower.endswith(".parquet"):
        return pd.read_parquet(io.BytesIO(raw))

    if lower.endswith(".jsonl"):
        rows = []
        for line in raw.decode("utf-8", errors="ignore").splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
        return pd.DataFrame(rows) if rows else None

    if lower.endswith(".json"):
        text = raw.decode("utf-8", errors="ignore").strip()
        if not text:
            return None
        try:
            obj = json.loads(text)
        except Exception:
            return None

        if isinstance(obj, list):
            return pd.DataFrame(obj)

        if isinstance(obj, dict):
            if "rows" in obj and isinstance(obj["rows"], list):
                return pd.DataFrame(obj["rows"])
            if "data" in obj and isinstance(obj["data"], list):
                return pd.DataFrame(obj["data"])
            return pd.DataFrame([obj])

    return None


def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = " ".join(text.split())
    return text.strip()


def pick_first(row: Dict, candidates: List[str], default: str = "") -> str:
    for c in candidates:
        if c in row:
            val = row[c]
            if pd.notna(val):
                val = clean_text(val)
                if val:
                    return val
    return default


def normalize_maud_row(row: Dict, source_uri: str, row_idx: int) -> Optional[Dict]:
    text = pick_first(
        row,
        [
            "section_text",
            "text",
            "clean_text",
            "cleaned_text",
            "content",
            "body",
            "body_norm",
            "section_content",
            "clause_text",
        ],
    )
    if not text:
        return None

    doc_id = pick_first(
        row,
        [
            "doc_id",
            "document_id",
            "contract_id",
            "agreement_id",
            "file_id",
            "file_name",
            "filename",
            "id",
        ],
        default=f"maud_doc_{row_idx}",
    )

    section_title = pick_first(
        row,
        [
            "section_title",
            "section",
            "heading",
            "title",
            "header",
            "clause_type",
            "label",
        ],
        default="",
    )

    return {
        "dataset": "maud",
        "doc_id": doc_id,
        "section_title": section_title,
        "text": text,
        "source_uri": source_uri,
        "row_idx": row_idx,
    }


def chunk_text(text: str, tokenizer, max_tokens: int, overlap_tokens: int) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )["input_ids"]

    if not token_ids:
        return []

    if len(token_ids) <= max_tokens:
        chunk = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        return [chunk] if chunk else []

    chunks = []
    step = max_tokens - overlap_tokens
    for start in range(0, len(token_ids), step):
        end = start + max_tokens
        piece = token_ids[start:end]
        if not piece:
            continue
        chunk = tokenizer.decode(piece, skip_special_tokens=True).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(token_ids):
            break
    return chunks


def make_vector_key(
    dataset: str,
    doc_id: str,
    source_uri: str,
    row_idx: int,
    chunk_idx: int,
    chunk_text_value: str,
) -> str:
    base = f"{dataset}|{doc_id}|{source_uri}|row={row_idx}|chunk={chunk_idx}"
    base_hash = hashlib.md5(base.encode("utf-8")).hexdigest()[:20]
    txt_hash = hashlib.md5(chunk_text_value.encode("utf-8")).hexdigest()[:12]
    return f"{dataset}::{base_hash}::{txt_hash}"


def chunked(lst: List, n: int) -> Iterable[List]:
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def ensure_vector_bucket_and_index():
    bucket_exists = True
    try:
        s3vectors.get_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)
    except ClientError as e:
        if e.response["Error"]["Code"] in {"NotFoundException", "ResourceNotFoundException"}:
            bucket_exists = False
        else:
            raise

    if not bucket_exists:
        s3vectors.create_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)

    index_exists = True
    try:
        s3vectors.get_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in {"NotFoundException", "ResourceNotFoundException"}:
            index_exists = False
        else:
            raise

    if not index_exists:
        s3vectors.create_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            dataType=DATA_TYPE,
            dimension=INDEX_DIMENSION,
            distanceMetric=DISTANCE_METRIC,
        )


def existing_keys(keys: List[str]) -> set:
    found = set()
    for batch in chunked(keys, GET_EXISTING_BATCH_SIZE):
        try:
            resp = s3vectors.get_vectors(
                vectorBucketName=TARGET_VECTOR_BUCKET,
                indexName=TARGET_INDEX_NAME,
                keys=batch,
            )
            for v in resp.get("vectors", []):
                k = v.get("key")
                if k:
                    found.add(k)
        except ClientError:
            pass
    return found


def put_vectors(vectors: List[Dict]):
    for batch in chunked(vectors, PUT_BATCH_SIZE):
        s3vectors.put_vectors(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            vectors=batch,
        )


def flush_pending(model, pending_rows: List[Dict], stats: Dict[str, int]) -> List[Dict]:
    if not pending_rows:
        return []

    unique_rows = {}
    for r in pending_rows:
        unique_rows[r["key"]] = r
    pending_rows = list(unique_rows.values())

    keys = [r["key"] for r in pending_rows]
    already = existing_keys(keys)

    to_insert = [r for r in pending_rows if r["key"] not in already]
    stats["skipped_existing"] += len(pending_rows) - len(to_insert)
    stats["attempted_chunks"] += len(pending_rows)

    if not to_insert:
        return []

    texts = [r["chunk_text"] for r in to_insert]
    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=128,
        show_progress_bar=False,
    )

    payload = []
    seen_payload_keys = set()

    for row, emb in zip(to_insert, embeddings):
        if row["key"] in seen_payload_keys:
            continue
        seen_payload_keys.add(row["key"])

        payload.append(
            {
                "key": row["key"],
                "data": {"float32": [float(x) for x in emb]},
                "metadata": {
                    "dataset": "maud",
                    "doc_id": row["doc_id"][:500],
                    "section_title": row["section_title"][:500],
                    "source_uri": row["source_uri"][:1000],
                },
            }
        )

    if payload:
        put_vectors(payload)
        stats["inserted_chunks"] += len(payload)

    return []


# =========================
# MAIN
# =========================
def main():
    print("Loading embedding model...")
    model = SentenceTransformer(EMBED_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

    print("Ensuring target vector bucket and index exist...")
    ensure_vector_bucket_and_index()

    print("Discovering MAUD files...")
    keys = []
    for prefix in SOURCE_PREFIXES:
        keys.extend(list(list_s3_keys(SOURCE_BUCKET, prefix)))

    seen = set()
    filtered_keys = []
    for k in keys:
        if k not in seen:
            seen.add(k)
            filtered_keys.append(k)

    print(f"Files found: {len(filtered_keys)}")

    pending_rows = []
    stats = {
        "files_seen": 0,
        "rows_seen": 0,
        "normalized_rows": 0,
        "attempted_chunks": 0,
        "inserted_chunks": 0,
        "skipped_existing": 0,
    }

    for key in tqdm(filtered_keys, desc="Processing MAUD files"):
        stats["files_seen"] += 1

        df = read_dataframe_from_s3(SOURCE_BUCKET, key)
        if df is None or df.empty:
            continue

        source_uri = f"s3://{SOURCE_BUCKET}/{key}"
        rows = df.fillna("").to_dict(orient="records")

        for i, row in enumerate(rows):
            stats["rows_seen"] += 1

            norm = normalize_maud_row(row, source_uri, i)
            if not norm:
                continue

            stats["normalized_rows"] += 1
            chunks = chunk_text(norm["text"], tokenizer, MAX_TOKENS, OVERLAP_TOKENS)
            if not chunks:
                continue

            for chunk_idx, chunk_text_value in enumerate(chunks):
                vector_key = make_vector_key(
                    dataset="maud",
                    doc_id=norm["doc_id"],
                    source_uri=norm["source_uri"],
                    row_idx=norm["row_idx"],
                    chunk_idx=chunk_idx,
                    chunk_text_value=chunk_text_value,
                )
                pending_rows.append(
                    {
                        "key": vector_key,
                        "doc_id": norm["doc_id"],
                        "section_title": norm["section_title"],
                        "source_uri": norm["source_uri"],
                        "chunk_text": chunk_text_value,
                    }
                )

            if len(pending_rows) >= EMBED_BATCH_CHUNKS:
                pending_rows = flush_pending(model, pending_rows, stats)
                print(
                    f"Inserted={stats['inserted_chunks']}  "
                    f"Skipped={stats['skipped_existing']}  "
                    f"Attempted={stats['attempted_chunks']}"
                )

    if pending_rows:
        pending_rows = flush_pending(model, pending_rows, stats)

    print("\nDone.")
    print(f"Files seen        : {stats['files_seen']}")
    print(f"Rows seen         : {stats['rows_seen']}")
    print(f"Normalized rows   : {stats['normalized_rows']}")
    print(f"Attempted chunks  : {stats['attempted_chunks']}")
    print(f"Inserted chunks   : {stats['inserted_chunks']}")
    print(f"Skipped existing  : {stats['skipped_existing']}")


if __name__ == "__main__":
    main()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Inserted=105207  Skipped=0  Attempted=105207
Inserted=105468  Skipped=0  Attempted=105468
Inserted=105725  Skipped=0  Attempted=105725
Inserted=105981  Skipped=0  Attempted=105981
Inserted=106269  Skipped=0  Attempted=106269
Inserted=106525  Skipped=0  Attempted=106525
Inserted=106787  Skipped=0  Attempted=106787
Inserted=107044  Skipped=0  Attempted=107044
Inserted=107300  Skipped=0  Attempted=107300
Inserted=107558  Skipped=0  Attempted=107558
Inserted=107818  Skipped=0  Attempted=107818
Inserted=108080  Skipped=0  Attempted=108080
Inserted=108400  Skipped=0  Attempted=108400
Inserted=108677  Skipped=0  Attempted=108677
Inserted=108940  Skipped=0  Attempted=108940
Inserted=109234  Skipped=0  Attempted=109234
Inserted=109491  Skipped=0  Attempted=109491
Inserted=109747  Skipped=0  Attempted=109747
Inserted=110030  Skipped=0  Attempted=110030
Inserted=110300  Skipped=0  Attempted=110300
Inserted=110563  Skipped=0  Attempted=110563
Inserted=110854  Skipped=0  Attempted=110854
Inserted=1

Processing MAUD files:  80%|████████  | 4/5 [5:05:34<1:17:55, 4675.19s/it]

Inserted=140276  Skipped=0  Attempted=140276
Inserted=140537  Skipped=0  Attempted=140537
Inserted=140794  Skipped=0  Attempted=140794
Inserted=141050  Skipped=0  Attempted=141050
Inserted=141338  Skipped=0  Attempted=141338
Inserted=141594  Skipped=0  Attempted=141594
Inserted=141856  Skipped=0  Attempted=141856
Inserted=142113  Skipped=0  Attempted=142113
Inserted=142369  Skipped=0  Attempted=142369
Inserted=142627  Skipped=0  Attempted=142627
Inserted=142887  Skipped=0  Attempted=142887
Inserted=143149  Skipped=0  Attempted=143149
Inserted=143469  Skipped=0  Attempted=143469
Inserted=143746  Skipped=0  Attempted=143746
Inserted=144009  Skipped=0  Attempted=144009
Inserted=144303  Skipped=0  Attempted=144303
Inserted=144560  Skipped=0  Attempted=144560
Inserted=144816  Skipped=0  Attempted=144816
Inserted=145099  Skipped=0  Attempted=145099
Inserted=145369  Skipped=0  Attempted=145369
Inserted=145632  Skipped=0  Attempted=145632
Inserted=145923  Skipped=0  Attempted=145923
Inserted=1

Processing MAUD files: 100%|██████████| 5/5 [6:24:58<00:00, 4619.73s/it]

Inserted=175345  Skipped=0  Attempted=175345

Done.
Files seen        : 5
Rows seen         : 46430
Normalized rows   : 29410
Attempted chunks  : 175345
Inserted chunks   : 175345
Skipped existing  : 0
